# Mamba SOH — LFP v2.1 · multi-temperature retrain

`soh_mamba_v2.0-lfp.pth` train trên Severson, mà **toàn bộ Severson chạy trong một buồng 30 °C**.
Hệ quả đo được trên pin LFP hoàn toàn khoẻ, chỉ đổi nhiệt độ:

| Nhiệt độ | SOH đọc ra | action_code |
|---|---|---|
| 10 °C | 85.7% | **REPLACE_IMMEDIATELY (P1)** |
| 25–35 °C | 100% | MONITOR |
| 45 °C | 92.4% | SCHEDULE_MAINTENANCE (P3) |

Đổi severity cờ OOD **không** sửa được — bản thân SOH sai. Chỉ có dữ liệu ở nhiệt độ khác mới sửa được.

## v2.1 thêm gì

**Sandia National Laboratories** (Preger et al. 2020, JES 167 120532) — 18 pin LFP ở **15/25/35 °C**,
cùng đúng loại cell với Severson (**A123 APR18650M1A, 1.1 Ah**) nên gộp là hợp lệ.

| | v2.0-lfp | **v2.1-lfp** |
|---|---|---|
| Nguồn | Severson 124 cell @30 °C | + SNL 18 cell @15/25/35 °C |
| Cụm nhiệt | `(30.0,)` | `(15.0, 25.0, 30.0, 35.0)` |
| Hết cờ OOD giả | 25–35 °C | **10–40 °C** |
| C-rate xả trong train | chỉ 4C | + **0.5C** (RPT) — gần chế độ solar thật hơn nhiều |
| `cycle_count_norm` | 2300 | **4600** (pin SNL chạy tới 4569 chu kỳ) |

Số liệu SNL đã kiểm chứng thật (chạy local 2026-08-11, không phải ước lượng):
**27.347 window train · 1.541 val · 2.161 test · SOH 73,2 → 97,9%**.

---

## Checklist trước khi Run All

| # | Việc | Bắt buộc |
|---|------|---------|
| 1 | Settings → Accelerator → **GPU T4 x2** | ✅ (KHÔNG chọn P100 — Kaggle PyTorch bỏ sm_60) |
| 2 | Settings → **Internet: On** | ✅ nếu để notebook tự tải SNL từ Zenodo |
| 3 | **+ Add Data** → `rickandjoe/mit-battery-degradation-dataset` | ✅ (Severson `.mat`, 8 GB) |
| 4 | Add-ons → Secrets → `GITHUB_TOKEN` | ✅ nếu repo private |
| 5 | Đã `git push` nhánh retrain lên GitHub | ✅ cell 3 sẽ chặn nếu chưa |
| 6 | Chạy bằng **Save Version → Save & Run All (Commit)** | ✅ **KHÔNG bấm Run All** |

> ### ⚠️ Run All KHÔNG giữ được output
>
> `Run All` chạy trong tab trình duyệt. Đóng tab / mất mạng / idle → session chết và
> **`/kaggle/working` bị xoá sạch**, mất cả 4 artifact sau nhiều giờ train.
>
> Dùng **Save Version → Save & Run All (Commit)**: chạy ở container nền, đóng máy được,
> xong thì lấy file ở tab **Output** của version.
>
> Ở chế độ Commit, **một cell raise là cả version fail và không có output**. Vì vậy
> notebook này đóng gói artifact ở cell 9 — TRƯỚC mọi bước chẩn đoán — và cell chẩn đoán
> chỉ in cảnh báo, không `assert`. Mọi `assert` đều nằm ở 4 cell đầu, fail trong 2 phút.

> SNL **không có trên Kaggle** (đã tra bằng CLI). Notebook tự tải 115 MB từ Zenodo (CC-BY-4.0,
> không cần đăng nhập). Nếu tắt Internet thì phải tự upload SNL.zip thành Kaggle dataset và attach.


## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chua bat GPU: Settings -> Accelerator -> GPU T4 x2'
name = torch.cuda.get_device_name(0)
print('GPU:', name)
assert 'P100' not in name, 'P100 khong tuong thich PyTorch Kaggle (sm_60) - doi sang GPU T4 x2'

## 2 — Clone repo

In [ ]:
import subprocess, os
BRANCH = 'feat/lfp-v21-multitemp'      # <-- doi cho khop nhanh dang lam
REPO   = '/kaggle/working/ai-module'
url = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('Khong co GITHUB_TOKEN secret -> thu public clone:', e)

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, REPO], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git', 'log', '-1', '--format=%h %ci %s']).decode())

## 3 — Chặn bẫy "code trên GitHub chưa mới"

Bẫy này đã làm mất ~11 giờ ở GH-67: sửa notebook nhưng quên push code.
Fail ở đây → về máy `git push` rồi **Restart & Run All** (phải xoá session, vì cell 2 bỏ qua
clone khi thư mục đã tồn tại).

In [ ]:
import pathlib

checks = {
    'scripts/preprocess_snl.py': ['load_snl_dir', '--severson-dir', 'BATTERYARCHIVE_COLUMNS',
                                  'MAX_DT_SECONDS_DEFAULT', '--cycle-count-norm',
                                  '--snl-cycle-stride', '--artifact-version',
                                  'temp_mean_c'],
    'scripts/preprocess_lfp.py': ['cycles_to_windows', '_longest_discharge_segment',
                                  'MAX_DISCHARGE_SECONDS', 'return_meta'],
    'scripts/eval_soh_by_temp.py': ['temp_mean_c', 'cell_ids'],
    'scripts/train.py':          ['--feature-scaler-version', '--mamba-out', '--iso-out',
                                  '--balance-bands', '--balance-temp-bins'],
    'src/core/config.py':        ['LFP_CYCLE_COUNT_NORM', 'LFP_NOMINAL_CAPACITY_AH',
                                  'LFP_TEMPERATURE_TRAIN_CLUSTERS', 'SPECTRAL_FEAT_DIM'],
}
missing = []
for path, needles in checks.items():
    p = pathlib.Path(path)
    if not p.exists():
        missing.append(path + ': FILE KHONG TON TAI')
        continue
    text = p.read_text(encoding='utf-8')
    for n in needles:
        if n not in text:
            missing.append(f'{path}: thieu {n!r}')
assert not missing, 'Code tren GitHub CHUA MOI:\n  ' + '\n  '.join(missing)
print('OK - code clone ve dung ban moi')

## 4 — Dependencies

In [ ]:
%pip install -q h5py scipy scikit-learn joblib pandas
import h5py, scipy, sklearn
print('h5py', h5py.__version__, '| scipy', scipy.__version__, '| sklearn', sklearn.__version__)

## 5 — Lấy dữ liệu

**Severson** — phải attach qua `+ Add Data` (8 GB, không tải trong notebook được).

**SNL** — thứ tự ưu tiên:
1. dataset đã attach (tìm `SNL_18650_LFP*.pkl` hoặc `SNL.zip` trong `/kaggle/input`)
2. tải thẳng từ Zenodo (115 MB, ~30 giây, cần Internet: On)

Nguồn: `https://zenodo.org/records/19688272/files/SNL.zip` — bản chuẩn hoá của BatteryLife,
CC-BY-4.0. batteryarchive.org có 30 pin LFP nhưng đã bỏ link tải thẳng; bản này có 18 pin.

In [ ]:
import os, glob, subprocess

# --- Severson ---------------------------------------------------------------
mats = glob.glob('/kaggle/input/**/*batch*.mat', recursive=True)
assert mats, ('Khong thay *batch*.mat. + Add Data -> '
              'rickandjoe/mit-battery-degradation-dataset')
SEVERSON_DIR = os.path.dirname(mats[0])
print('SEVERSON_DIR:', SEVERSON_DIR)
for f in sorted(mats):
    print('   ', os.path.basename(f), '(%.2f GB)' % (os.path.getsize(f) / 1e9))

# --- SNL --------------------------------------------------------------------
SNL_SRC = None
pkls = glob.glob('/kaggle/input/**/SNL_18650_LFP*.pkl', recursive=True)
zips = glob.glob('/kaggle/input/**/SNL.zip', recursive=True)
if pkls:
    SNL_SRC = os.path.dirname(pkls[0]);  print('\nSNL tu dataset attach:', SNL_SRC, f'({len(pkls)} pkl)')
elif zips:
    SNL_SRC = zips[0];                   print('\nSNL tu zip attach:', SNL_SRC)
else:
    SNL_SRC = '/kaggle/working/SNL.zip'
    if not os.path.exists(SNL_SRC):
        print('\nTai SNL.zip tu Zenodo (115 MB)...')
        subprocess.run(['curl', '-sSL', '-o', SNL_SRC,
                        'https://zenodo.org/records/19688272/files/SNL.zip?download=1'], check=True)
    size = os.path.getsize(SNL_SRC)
    assert size > 50e6, (f'SNL.zip chi {size} byte -> tai that bai. '
                         f'Bat Settings -> Internet: On, hoac tu upload dataset.')
    print('   OK: %.1f MB' % (size / 1e6))

import zipfile
if zipfile.is_zipfile(SNL_SRC):
    n = len([x for x in zipfile.ZipFile(SNL_SRC).namelist() if 'LFP' in x and x.endswith('.pkl')])
    assert n >= 15, f'Chi thay {n} file LFP trong zip - file hong?'
    print('   %d cell LFP trong archive' % n)

## 6 — Preprocess (SNL + Severson gộp trước khi scale)

Gộp **trước** khi scale là bắt buộc: scale riêng từng nguồn rồi mới gộp sẽ đặt hai tập lên hai
thang khác nhau, model học rác mà không có lỗi nào báo.

Tham số đã kiểm chứng trên dữ liệu thật:

- `--max-dt-seconds 30` — SNL log ở **hai tần số**: chu kỳ RPT dt=10 s (~745 mẫu đoạn xả, dùng
  được) và chu kỳ già hoá thường dt=120 s (xả 3C chỉ còn **10 mẫu** < window 30, không dùng được).
  Không lọc thì ~95% chu kỳ rơi vào nhánh "short" và scaler mất sạch vùng 2C/3C.
- `--snl-cycle-stride 1` — sau khi lọc mỗi pin chỉ còn 53–151 chu kỳ, không có gì để bớt.
- `--cycle-stride 3` — cho Severson, giữ nguyên như v2.0.
- `--cycle-count-norm 4600` — pin SNL chạy tới **4569** chu kỳ; để 2300 thì nửa dữ liệu clip về
  1.0, trục lão hoá phẳng đúng chỗ pin già nhất.
- `--phase discharge`, `--soh-clip 100`, `--soc-mode cycle` — giữ nguyên v2.0, đã chứng minh đúng.

Val/test mặc định đã cắm sẵn trong script: `25C_…3C_c` (val) và `35C_…2C_b` (test) — hai pin
**xuống dưới 80% SOH**, mỗi pin một mức nhiệt, cộng thêm ~4% pin Severson để giữ số 30 °C so
sánh được với v2.0. Pin 15 °C duy nhất **cố ý để trong train**.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess_snl.py \
    --snl-dir "{SNL_SRC}" \
    --severson-dir "{SEVERSON_DIR}" \
    --output-dir data/processed_lfp \
    --cycle-stride 3 --snl-cycle-stride 1 --max-dt-seconds 30 \
    --cycle-count-norm 4600 --artifact-version 2.1-lfp \
    --phase discharge --time-unit minutes --soh-clip 100 --soc-mode cycle

## 7 — Cổng kiểm kê: phủ sóng (nhiệt độ × SOH)

**Đọc bảng này trước khi bấm train.** Đây đúng là lỗi đã gây ra GH-88: train thiếu một dải SOH ở
một mức nhiệt → model ngoại suy lệch ngay tại ngưỡng EOL 80%, không test nào bắt được vì kết quả
vẫn "hợp lệ".

Ô `.` ở dải **80–90%** của một cụm nhiệt = model **đoán mò đúng tại vùng ra quyết định thay pin**
ở nhiệt độ đó.

In [ ]:
import torch, numpy as np, collections, joblib

TEMP_IDX = 2   # BASE_FEATURES = [voltage, current, temperature, time]
BANDS = [(100, 105), (95, 100), (90, 95), (85, 90), (80, 85), (70, 80), (50, 70), (0, 50)]

# Doi nhiet do da scale -> °C that, de bang doc duoc
sc = joblib.load('models/weights/scaler_lfp.pkl')['scaler']
lo, hi = sc.data_min_[TEMP_IDX], sc.data_max_[TEMP_IDX]
print('scaler temperature range: [%.2f, %.2f] °C' % (lo, hi))

for split in ['train', 'val', 'test']:
    d = torch.load('data/processed_lfp/%s.pt' % split, weights_only=False)
    X, y = d['X'].numpy(), d['y'].numpy()
    t_c = X[:, :, TEMP_IDX].mean(axis=1) * (hi - lo) + lo      # -> °C
    t_g = np.round(t_c / 5.0) * 5.0                            # gom cum 5 °C
    groups = sorted(set(t_g))
    print('\n=== %s: %d window ===' % (split, len(y)))
    hdr = 'SOH band  ' + ''.join('%9.0fC' % g for g in groups) + '     TONG'
    print(hdr); print('-' * len(hdr))
    for b_lo, b_hi in BANDS:
        m = (y >= b_lo) & (y < b_hi)
        if not m.any():
            continue
        row = '%3d-%3d  ' % (b_lo, b_hi)
        for g in groups:
            n = int((m & (t_g == g)).sum())
            row += '%10d' % n if n else '         .'
        print(row + '%9d' % int(m.sum()))
    tot = '   TONG  ' + ''.join('%10d' % int((t_g == g).sum()) for g in groups)
    print(tot + '%9d' % len(y))
print('\n[i] Dau "." = KHONG CO MAU o o do -> vung mu cua model.')

## 8 — Train

Kiến trúc + hyperparameter **giữ nguyên v2.0**. Đổi cùng lúc cả data lẫn hyperparameter thì số
xấu đi sẽ không quy được trách nhiệm.

`--balance-bands --balance-temp-bins 12` đặc biệt quan trọng ở v2.1: nó cân theo tần suất
**(nhiệt độ × dải SOH)**, đúng thứ đang mất cân bằng (140 pin Severson @30 °C vs 6 pin @35 °C
vs 1 pin @15 °C).

> `12` không phải số đẹp tuỳ chọn. Bin được chia trên kênh nhiệt độ **đã scale**, nên bề rộng
> mỗi bin = dải_nhiệt/n. Dải thật đo được là ~44,6 °C, mà các cụm chỉ cách nhau 5 °C ⇒ mặc định
> 3 bin (14,9 °C/bin) và cả 6 bin (7,4 °C/bin) đều **gộp 30 °C với 35 °C**, khiến 6 pin SNL @35 °C
> không bao giờ được nâng trọng số. 12 bin = 3,7 °C/bin mới tách hết. Có test khoá:
> `tests/test_balance_weights.py::test_twelve_bins_separate_every_real_cluster`.

`--jitter` để mặc định 0 — GH-67 lần 7 đã chứng minh `0.01` làm tệ đi (nhiễu/tín hiệu ở kênh
current lên tới 4608%).

> Thời gian dự kiến: ~35k window train → ~1.1k step/epoch × 50 epoch trên T4. Nếu quá 9 giờ,
> giảm bằng `--cycle-stride 5` cho Severson chứ **đừng** đụng `--snl-cycle-stride`.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py \
    --data-dir data/processed_lfp \
    --epochs 50 \
    --balance-bands --balance-temp-bins 12 \
    --swa \
    --log-dir logs/training \
    --mamba-out models/weights/soh_mamba_v2.1-lfp.pth \
    --iso-out models/weights/isolation_forest_v2.1-lfp.pkl \
    --model-version 2.1-lfp \
    --feature-scaler-version 2.1-lfp

## 9 — Đóng gói artifact

**Chạy TRƯỚC mọi bước chẩn đoán.** GH-67 lần 4: một `assert` chẩn đoán fail đã giết notebook
*sau khi train xong 4 tiếng* → mất trắng. Zip trước, soi sau.

In [ ]:
import shutil, os
OUT = '/kaggle/working/lfp_v21_artifacts'
os.makedirs(OUT, exist_ok=True)
need = ['models/weights/soh_mamba_v2.1-lfp.pth',
        'models/weights/isolation_forest_v2.1-lfp.pkl',
        'models/weights/scaler_lfp.pkl',
        'models/weights/feature_scaler_lfp.pkl']
for p in need:
    assert os.path.exists(p), 'Thieu artifact: ' + p
    shutil.copy(p, OUT)
    print('  +', os.path.basename(p), '(%.0f KB)' % (os.path.getsize(p) / 1024))
shutil.make_archive(OUT, 'zip', OUT)
print('\nlfp_v21_artifacts.zip da nam trong /kaggle/working')
print('  - Commit mode : lay o tab Output cua version sau khi chay xong')
print('  - Run All     : tai NGAY o panel Output ben phai, dung doi het notebook')

## 10 — Nghiệm thu: MAE / RMSE + bóc tách theo nhiệt độ

**Xem theo dải trước, MAE tổng sau** — MAE tổng bị dải đông mẫu chi phối nên luôn trông đẹp.

Bốn con số quyết định:
1. **MAE / RMSE tổng** — mục tiêu `< 2.0%` / `< 3.0%`
2. **bias dải 70–80%** — dương nghĩa là pin 75% bị báo thành 85% → **bỏ sót pin cần thay**
3. **MAE theo nhiệt độ** — MAE ở 15/35 °C cao hơn hẳn 30 °C nghĩa là thêm SNL **chưa** đạt mục đích
4. **`bias@healthy`** — bias của window có SOH thật ≥ 92%, tách theo nhiệt độ. Đây là **phép đo
   trực tiếp của bug**: pin khoẻ ở nhiệt độ lạ/nóng bị đọc thấp đi bao nhiêu điểm. Càng gần 0 càng tốt.

Cell này **chỉ cảnh báo, không `assert`** — chẩn đoán không được phép giết một lần chạy nhiều giờ.

In [ ]:
import glob, os, re, sys, torch, numpy as np, joblib
sys.path.insert(0, '.')
sys.path.insert(0, 'scripts')
from src.core.config import D_MODEL, D_STATE, INPUT_FEATURES, SPECTRAL_FEAT_DIM
from src.models.soh_predictor import MambaSOHPredictor
from train import evaluate          # dung CHINH ham train.py da dung -> khong lech convention

PREV = {'mae': 1.2899, 'rmse': 1.8935}   # v2.0-lfp, lan train tot nhat
def chk(ok, msg): print(('  [OK] ' if ok else '  [!]  ') + msg)

ck = torch.load('models/weights/soh_mamba_v2.1-lfp.pth', map_location='cpu', weights_only=False)
mae, rmse = ck['test_mae'], ck['test_rmse']

print('=== 1. MAE / RMSE TONG THE ===')
chk(mae  < 2.0, 'test MAE  = %.4f %%   (target < 2.0 | v2.0-lfp = %.4f)' % (mae, PREV['mae']))
chk(rmse < 3.0, 'test RMSE = %.4f %%   (target < 3.0 | v2.0-lfp = %.4f)' % (rmse, PREV['rmse']))

print('\n=== 2. PER-BAND (tu train log) ===')
logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
if logs:
    for ln in open(logs[-1], encoding='utf-8'):
        if re.search(r'SOH\s+\d+-\d+', ln):
            print('  ', ln.split('INFO')[-1].strip())
else:
    print('  (khong tim thay train log)')

print('\n=== 3. BOC TACH THEO NHIET DO ===')
d = torch.load('data/processed_lfp/test.pt', weights_only=False)
X, Xf, y_t = d['X'], d['X_feat'], d['y']
model = MambaSOHPredictor(input_features=INPUT_FEATURES, d_model=D_MODEL,
                          d_state=D_STATE, feat_dim=SPECTRAL_FEAT_DIM)
model.load_state_dict(ck['model_state_dict'])
res = evaluate(model, X, Xf, y_t, torch.device('cpu'))
pred, y = res['pred'].numpy(), y_t.numpy()

# Chan sai lech convention (model xuat SOH/100, evaluate() nhan x100). Neu so tinh
# lai o day khong khop so trong checkpoint thi moi bang ben duoi deu vo nghia.
chk(abs(res['mae'] - mae) < 0.01,
    'MAE tinh lai = %.4f khop checkpoint %.4f' % (res['mae'], mae))

sc = joblib.load('models/weights/scaler_lfp.pkl')['scaler']
lo, hi = sc.data_min_[2], sc.data_max_[2]
t_c = X[:, :, 2].numpy().mean(axis=1) * (hi - lo) + lo
t_g = np.round(t_c / 5.0) * 5.0

print('  %8s %8s %9s %9s | %11s %13s' % ('temp', 'n', 'MAE', 'bias', 'n(SOH>=92)', 'bias@healthy'))
for g in sorted(set(t_g)):
    m = t_g == g
    if m.sum() < 30:
        continue
    err = pred[m] - y[m]
    h = m & (y >= 92)
    hb = '%+13.3f' % (pred[h] - y[h]).mean() if h.sum() >= 20 else '%13s' % 'n/a'
    print('  %7.0fC %8d %9.3f %+9.3f | %11d %s'
          % (g, m.sum(), np.abs(err).mean(), err.mean(), h.sum(), hb))

worst = max((np.abs(pred[t_g == g] - y[t_g == g]).mean()
             for g in set(t_g) if (t_g == g).sum() >= 30), default=0.0)
chk(worst < 3.0, 'MAE cua cum nhiet TE NHAT = %.3f%% (nen < 3.0)' % worst)
print('\n[i] `bias@healthy` la phep do truc tiep cua bug goc: pin KHOE bi doc THAP di bao nhieu')
print('    diem tai nhiet do do. Con am nhieu (vd -8) = bug van con.')

## 10b — Bóc tách theo **cell** (ô 10 không làm được)

Ô 10 dựng lại nhiệt độ bằng cách nghịch đảo MinMax trên cột đã scale — chính xác, vì MinMax là
phép affine. Nhưng **danh tính cell thì không nằm trong `X`**, nên không có cách nào biết cell
nào đang sai nhiều.

Việc đó quan trọng cho bước dọn dữ liệu: `scaler_lfp.pkl` đang có `temperature min = 0.0 °C` và
`voltage min = 1.889 V/cell` — cả hai bất khả thi về vật lý. Xếp hạng MAE theo cell là cách
nhanh nhất để tìm ra cell nào mang dữ liệu hỏng đó.

`scripts/preprocess_snl.py` giờ ghi `cell_idx` / `cell_ids` / `temp_mean_c` / `cycle_idx` vào
`.pt`. Nếu ô dưới báo thiếu khoá thì split được sinh trước thay đổi này — chạy lại ô preprocess
(**không cần train lại**, script chỉ chấm điểm checkpoint sẵn có).

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/eval_soh_by_temp.py     --data-dir data/processed_lfp     --split test     --weights models/weights/soh_mamba_v2.1-lfp.pth

## 11 — Config phải sửa khi commit artifact

In ra đúng diff cần áp vào `src/core/config.py`. **Không sửa trước** — khai cụm nhiệt mới trong
khi weight vẫn là v2.0 sẽ tắt cờ OOD cho một model thật sự không xử lý được 15 °C, tức là biến
một cảnh báo đúng thành im lặng sai.

In [ ]:
import joblib
meta = joblib.load('models/weights/scaler_lfp.pkl')
clusters = tuple(float(c) for c in meta.get('temperature_clusters', []))
print('Sua src/core/config.py:')
print()
print('  LFP_MODEL_VERSION           = "2.1-lfp"        # was "2.0-lfp"')
print('  LFP_CYCLE_COUNT_NORM        = %.1f' % meta['cycle_count_norm'], '        # was 2300.0')
print('  LFP_TEMPERATURE_TRAIN_CLUSTERS = %s' % (clusters,), '  # was (30.0,)')
print()
print('Va cap nhat tests/test_models.py (3 test dang assert cum (30.0,)).')
print()
print('Commit CUNG LUC 4 artifact + 3 dong config tren. Tach ra 2 commit = mot commit sai.')

## 12 — Dọn output (artifact đã nằm trong zip)

In [ ]:
import os, shutil
os.chdir('/kaggle/working')
# Repo clone, thu muc artifact chua nen, va SNL.zip da tai ve deu KHONG can giu:
# moi thu can thiet nam trong lfp_v21_artifacts.zip. Don sach de output version gon.
shutil.rmtree('/kaggle/working/ai-module', ignore_errors=True)
shutil.rmtree('/kaggle/working/lfp_v21_artifacts', ignore_errors=True)
for f in ['SNL.zip']:
    p = os.path.join('/kaggle/working', f)
    if os.path.exists(p):
        os.remove(p)

left = sorted(os.listdir('/kaggle/working'))
print('Output con lai:', left)
assert 'lfp_v21_artifacts.zip' in left, 'MAT ARTIFACT - dung dong session, kiem tra lai cell 9'
mb = os.path.getsize('/kaggle/working/lfp_v21_artifacts.zip') / 1e6
print('lfp_v21_artifacts.zip = %.1f MB' % mb)
print()
print('LAY FILE O DAU:')
print('  Commit mode -> trang notebook -> tab Output -> tai lfp_v21_artifacts.zip')
print('  Run All     -> panel Output ben phai -> tai NGAY truoc khi session chet')